In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../out/csvfile/unknown/unknown_1-200_ver1.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../out/csvfile/unknown/unknown_1-200_ver1.csv'

In [20]:
df = df.sort_values(by="sigma", ascending=False, ignore_index=True)
for i in range(len(df)):
    print(df.loc[i, "smiles"], df.loc[i, "sigma"])

O=C(O)C1=C(C(=O)O)C(=CC=C1N(=O)=O)N(=O)=O 2.930214292816916
O=C(O)C=1C=C(C(=CC1C(=O)O)N(=O)=O)N(=O)=O 2.768663570936261
O=C(O)C1=C(O)C=C2C(=O)C=3C(O)=C(C(C(=O)C)=C(O)C3C(=O)C2=C1C(=O)O)CC 2.747609202775129
O=C(O)C1=CC=C(C(=C1C(=O)O)N(=O)=O)N(=O)=O 2.6599462859400376
O=C(O)C1=C(O)C=C2C(=O)C3=C(O)C(O)=C(C(O)=C3C(=O)C2=C1C(=O)O)C4=CC(=CC=C4O)CCN 2.5157570700488705
N#CC=1C=C(C(=O)O)C(=CC1C#N)C(=O)O 2.4919986473785443
O=C(O)C1=CC=C(C(NC(=O)C)=C1C(=O)O)N(=O)=O 2.454859356868141
O=C(O)C1=C(F)C(F)=C(O)C(F)=C1C(=O)O 2.4427838527918952
O=C(O)C=1C(Br)=CC=C(Br)C1C(=O)O 2.4418220834610778
O=C(O)C=1C(Cl)=CC=C(Cl)C1C(=O)O 2.437991329457852
O=C(O)C1=C(C(=O)O)C(=CC=C1C)C 2.4275593084400304
O=C(O)C=1C(F)=CC=C(F)C1C(=O)O 2.404307450353221
O=C(O)C=1C=C(C=C(C(=O)O)C1C(=O)O)N(=O)=O 2.402422281885402
O=C(O)C=1C(Cl)=CC(Cl)=C(Cl)C1C(=O)O 2.375176253843068
O=C(O)C=1C(OC)=CC=C(OC)C1C(=O)O 2.3652091178558603
O=C(O)C=1C(F)=CC=C(Br)C1C(=O)O 2.359820433005323
O=C(O)C=1C(Cl)=CC=C(C1C(=O)O)C(F)(F)F 2.3578149786040843


In [12]:
df[["cas", "sigma"]]

,cas,sigma
0,2300-16-5,2.930214
1,90348-28-0,2.768664
2,6219-66-5,2.747609
3,92971-15-8,2.659946
4,14597-16-1,2.515757
...,...,...
171,499793-28-1,1.891871
172,1868072-45-0,1.891701
173,4961-03-9,1.890635
174,835-58-5,1.889630


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib

df = pd.read_csv(
    r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\ver_3\out\save\value.csv"
)

df = df.dropna(how="any").reset_index(drop=True)

descriptors = [
    "dipole_moment_debye",
    "3and6",
    "lumo_ev",
    "homo_ev",
]


length = len(descriptors)
X = df[descriptors]
y = df["yield"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kernel = (
    ConstantKernel() * RBF([1.0] * length, length_scale_bounds=(1e-2, 1e10))
    + WhiteKernel()
)
model = GaussianProcessRegressor(
    kernel=kernel, n_restarts_optimizer=10, random_state=42
)
loo = LeaveOneOut()
y_true, y_pred = [], []
for i, (train_index, test_index) in enumerate(loo.split(X_scaled)):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    model.fit(X_train, y_train)

    y_pred.append(model.predict(X_test)[0])
    y_true.append(y_test.values[0])

    if i < 5:  # 最初の5回だけ表示
        print(f"Iter {i}: Kernel params = {model.kernel_}")
        # もしHOMO（4変数版の4番目）のLength Scaleが小さくなっている回があれば、
        # そのデータにとってはHOMOが「超重要」だということです。

mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

model.fit(X_scaled, y)



Iter 0: Kernel params = 14.9**2 * RBF(length_scale=[11.3, 14.3, 8.89, 7.14e+04]) + WhiteKernel(noise_level=3.52)
Iter 1: Kernel params = 14.7**2 * RBF(length_scale=[11.2, 13.9, 8.84, 4.63e+04]) + WhiteKernel(noise_level=3.44)
Iter 2: Kernel params = 13**2 * RBF(length_scale=[10.9, 11.6, 5.5, 4.64e+04]) + WhiteKernel(noise_level=2.98)
Iter 3: Kernel params = 14.3**2 * RBF(length_scale=[9.7, 15.1, 9.96, 1.48e+04]) + WhiteKernel(noise_level=3.33)
Iter 4: Kernel params = 14.4**2 * RBF(length_scale=[11.4, 13.9, 8.12, 4.1e+04]) + WhiteKernel(noise_level=3.54)


,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".",1**2 * RBF(le...noise_level=1)
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-10
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",10
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",False
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib

df = pd.read_csv(
    r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\ver_4\out\csvfile\model\samples_value.csv"
)

df = df.dropna(how="any").reset_index(drop=True)

descriptors = [
    "dipole_moment_debye",
    "3and6",
    "lumo_ev",
    "delta_g_hartree",
]


length = len(descriptors)
X = df[descriptors]
y = df["yield"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kernel = (
    ConstantKernel() * RBF([1.0] * length, length_scale_bounds=(1e-2, 1e10))
    + WhiteKernel()
)
model = GaussianProcessRegressor(
    kernel=kernel, n_restarts_optimizer=10, random_state=42
)
loo = LeaveOneOut()
y_true, y_pred = [], []
for train_index, test_index in loo.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    model.fit(X_train, y_train)

    y_pred.append(model.predict(X_test)[0])
    y_true.append(y_test.values[0])

mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

model.fit(X_scaled, y)

content = {"model": model, "scaler": scaler}

print(r2)
print(mae)


0.48393153299996783
1.7856651862243444


In [17]:
import glob
from pathlib import Path

name = "samples"

route = Path("../out/csvfile")
filepath = next(route.glob(f"**/{name}.csv"))

print(filepath)
print(type(filepath))
filepath = str(filepath)
print(filepath)
print(type(filepath))

..\out\csvfile\model\samples.csv
<class 'pathlib.WindowsPath'>
..\out\csvfile\model\samples.csv
<class 'str'>


In [1]:
import numpy
import sys

# numpyがどこにあるか表示
print(numpy.__file__)

# Pythonが自動で探しに行くフォルダ一覧を表示
# （この中に numpy のあるフォルダが含まれているはずです）
print(sys.path)

c:\Users\kkyom\anaconda3\envs\predict_activity\Lib\site-packages\numpy\__init__.py
['C:\\Program Files\\RevvitySignalsSoftware\\ChemDrawApplications\\ChemScript\\Lib', 'C:\\Users\\kkyom\\OneDrive\\デスクトップ\\python\\stock', 'c:\\Users\\kkyom\\anaconda3\\envs\\predict_activity\\python311.zip', 'c:\\Users\\kkyom\\anaconda3\\envs\\predict_activity\\DLLs', 'c:\\Users\\kkyom\\anaconda3\\envs\\predict_activity\\Lib', 'c:\\Users\\kkyom\\anaconda3\\envs\\predict_activity', '', 'c:\\Users\\kkyom\\anaconda3\\envs\\predict_activity\\Lib\\site-packages', 'c:\\Users\\kkyom\\anaconda3\\envs\\predict_activity\\Lib\\site-packages\\win32', 'c:\\Users\\kkyom\\anaconda3\\envs\\predict_activity\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\kkyom\\anaconda3\\envs\\predict_activity\\Lib\\site-packages\\Pythonwin']


In [8]:
import cclib

data = cclib.io.ccread(r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\data\in\known\sub_652-03-9.log")
nbo_charges = data.atomcharges["natural"]
print(nbo_charges)

[-0.31799  0.34547  0.34103 -0.3187   0.40316 -0.32838 -0.15895  0.84167
 -0.68392 -0.63608 -0.1504   0.84344 -0.68426 -0.63597  0.3979  -0.32979
  0.53586  0.5359 ]


In [9]:
def get_final_nbo_charges(logfile_path):
    with open(logfile_path, 'r') as f:
        lines = f.readlines()
    
    # ファイルの最後から逆順にNBOの要約を探す
    start_line = -1
    for i in range(len(lines)-1, -1, -1):
        if "Summary of Natural Population Analysis" in lines[i]:
            start_line = i
            break
    
    if start_line == -1:
        return None

    charges = []
    # 表のヘッダーを飛ばしてデータ行（Atom No Charge... の後）へ
    # 概ね Summary... の6行後からデータが始まります
    for line in lines[start_line+6:]:
        if "---" in line or "Total" in line: # 表の終わり
            break
        parts = line.split()
        if len(parts) >= 3:
            charges.append(float(parts[2]))
    return charges

# 使用例
final_charges = get_final_nbo_charges(r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\data\in\known\sub_652-03-9.log")

In [10]:
print(final_charges)

[-0.31591, 0.35584, 0.33016, -0.32111, 0.4162, -0.32379, -0.1797, 0.84039, -0.67774, -0.64762, -0.12747, 0.84606, -0.68202, -0.63676, 0.38437, -0.33278, 0.53628, 0.53559]


In [1]:
import calc.phthalicacid as pa
import pandas as pd

df = pd.read_excel(r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\ver_4\data\2nd_20260204.xlsx")
df = pa.detect_phthalic_acid(df)
df

,化合物名,cas,smiles,yield,cooh
0,フタル酸,88-99-3,O=C(C1=CC=CC=C1C(O)=O)O,13.320000,"[[8, 7, 10, 9, 16], [1, 2, 0, 11, 17]]"
1,"2,3-ナフタレンジカルボン酸",2169-87-1,O=C(C1=CC2=CC=CC=C2C=C1C(O)=O)O,11.800000,"[[12, 11, 14, 13, 22], [1, 2, 0, 15, 23]]"
2,ピロメリット酸,89-05-4,O=C(C1=CC(C(O)=O)=C(C(O)=O)C=C1C(O)=O)O,11.530000,"[[9, 8, 11, 10, 20], [5, 4, 7, 6, 19]]"
3,4-ニトロフタル酸,610-27-5,O=C(O)C1=CC=C(C=C1C(=O)O)N(=O)=O,15.650000,"[[9, 8, 10, 11, 19], [1, 3, 0, 2, 15]]"
4,3-ニトロフタル酸,603-11-2,O=C(O)C1=CC=CC(=C1C(=O)O)N(=O)=O,12.490000,"[[9, 8, 10, 11, 19], [1, 3, 0, 2, 15]]"
5,TE,te,O=C(C(C=C1)=C(C(O)=O)C=C1OC2=CC(OC3=CC=C(C(O)=...,4.180000,"[[6, 5, 8, 7, 47], [1, 2, 0, 44, 62]]"
6,DE,de,O=C(C(C=C1)=C(C(O)=O)C=C1OC2=CC=CC(OC3=CC(C(O)...,11.500000,"[[6, 5, 8, 7, 34], [1, 2, 0, 31, 45]]"
7,EE,ee,O=C(C1=CC(OC(C=C2)=CC=C2OC3=CC(OC4=CC=C(OC5=CC...,8.735648,"[[42, 41, 44, 43, 66], [1, 2, 0, 45, 67]]"
8,"アントラキノン-2,3-ジカルボン酸",27485-15-0,O=C1C2=C(C=C(C(O)=O)C(C(O)=O)=C2)C(C3=CC=CC=C3...,17.690000,"[[6, 5, 8, 7, 23], [10, 9, 12, 11, 24]]"
9,4-メトキシフタル酸,1885-13-8,COC1=CC=C(C(O)=O)C(C(O)=O)=C1,13.290000,"[[6, 5, 8, 7, 19], [10, 9, 12, 11, 20]]"


In [9]:
from rdkit import Chem

def test(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)  # 水素を追加
    phthalic_pattern = Chem.MolFromSmarts("c1ccc(C(=O)O)c(C(=O)O)c1")
    # フタル酸構造を持つかどうか
    matches_data = mol.GetSubstructMatches(phthalic_pattern)
    print("matches_data:", matches_data)
    oh_list = []
    cooh_list = []
    if matches_data:
        num = 0
        while True:
            # 2つ以上フタル酸構造を含まれていることを考慮して一つだけ検出
            matches = matches_data[num]
            for match in matches:
                atom = mol.GetAtomWithIdx(match)  # その番号の原子の情報
                symbol = atom.GetSymbol()  # その番号の原子
                # まだこの時点では水素の場所は検出されていないのでカルボン酸の水素を検出
                if symbol == "O":
                    neighbors = atom.GetNeighbors()  # その原子に隣接する原子の情報
                    elements = [n.GetSymbol() for n in neighbors]
                    if "H" in elements and "C" in elements:
                        for neighbor in neighbors:
                            if neighbor.GetSymbol() == "H":
                                oh_list.append((atom.GetIdx(), neighbor.GetIdx()))
            # 片側もしくは両側がエステルになっている場合を考慮
            if len(oh_list) < 2:
                num += 1
                oh_list = []
                continue
            idx = matches
            for match in matches:
                atom = mol.GetAtomWithIdx(match)
                symbol = atom.GetSymbol()
                # カルボン酸の炭素を起点に他の原子を検出
                if symbol == "C":
                    neighbors = atom.GetNeighbors()
                    elements = [n.GetSymbol() for n in neighbors]
                    if elements.count("O") >= 2:
                        for neighbor in neighbors:
                            if neighbor.GetSymbol() == "C":
                                c = neighbor.GetIdx()
                            elif neighbor.GetSymbol() == "O":
                                if len(neighbor.GetNeighbors()) == 1:
                                    o_double = neighbor.GetIdx()
                                elif len(neighbor.GetNeighbors()) == 2:
                                    for oh in oh_list:
                                        if oh[0] == neighbor.GetIdx():
                                            o_single = oh[0]
                                            h = oh[1]
                        # 自身の炭素原子、それに結合しているベンゼン環内の炭素原子、
                        # 炭素と2重結合している酸素原子、炭素と単結合している酸素原子、水素の順
                        cooh_list.append([atom.GetIdx(), c, o_double, o_single, h])
            break
    print(cooh_list)

test("O=C(C1=C(C(O)=O)C=CC(CC2=CC=C(C(O)=O)C(C(O)=O)=C2)=C1)O")
test("FC1=C(F)C(F)=C(C(O)=O)C(C(O)=O)=C1F")

matches_data: ((8, 9, 23, 2, 1, 0, 24, 3, 4, 6, 5, 7), (11, 12, 13, 14, 15, 17, 16, 18, 19, 21, 20, 22))
[[1, 2, 0, 24, 36], [4, 3, 6, 5, 25]]
matches_data: ((1, 2, 4, 6, 7, 9, 8, 10, 11, 13, 12, 14),)
[[7, 6, 9, 8, 16], [11, 10, 13, 12, 17]]


In [4]:
import pandas as pd

df = pd.read_csv(r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\ver_4\out\csvfile\descriptors\2nd_20260205.csv")
df = df.sort_values("r2", ascending=False)
print(df)
df.to_csv(r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\ver_4\out\csvfile\descriptors\2nd_20260205.csv", index=False)

                                       descriptor       mae        r2
1093         lumo_ev|dipole_moment_debye|sasa|hbd  1.159665  0.742361
689               homo_ev|gap_ev|h_nbo_charge|hbd  1.184148  0.740099
1495  omega|molecular_volume_A3|h_nbo_charge|logp  1.191498  0.726798
1478           omega|dipole_moment_debye|sasa|hbd  1.129345  0.711483
1542                  omega|h_nbo_charge|sasa|hbd  1.351712  0.699656
...                                           ...       ...       ...
1821         h_nbo_charge|o_nbo_charge|sasa|4and5  3.542620 -1.171583
1902                  o_nbo_charge|4and5|logp|hba  3.593357 -1.188514
1829         h_nbo_charge|o_nbo_charge|4and5|logp  3.671908 -1.191000
1689  dipole_moment_debye|o_nbo_charge|4and5|logp  3.879952 -1.322903
1429                       gap_ev|sasa|4and5|logp  3.338181 -1.500530

[1940 rows x 3 columns]


In [6]:
a = "1-100"
b,c = a.split("-")
print(type(b),c)

<class 'str'> 100


In [12]:
import pandas as pd

df = pd.read_csv(r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\ver_4\out\csvfile\predict\2nd_20260211_4.csv")
df = df[["smiles", "cas", "mu", "sigma"]]
df = df.sort_values("sigma", ascending=False, ignore_index=True)
for i in range(len(df)):
    print(df.loc[i, "smiles"])

O=C(O)C=1C(C(=O)O)=C(C(C(=O)O)=C(C(=O)O)C1C(F)(F)F)C(F)(F)F
O=C(O)C1=C(O)C=C2C(=O)C3=C(O)C(O)=C(C(O)=C3C(=O)C2=C1C(=O)O)C4=CC(=CC=C4O)CCN
O=C(O)C1=CC=C2C(=C1C(=O)O)C(C=3C=CC=CC3)(C)CC2(C)C
O=C(O)C1=C(OCC)C=2C=CC=CC2C(OCC)=C1C(=O)O
O=C(O)C=1C=C(C(=O)O)C(C(=O)O)=C(C(=O)O)C1C(=O)O
O=C(O)C1=CC=C(C=C1C(=O)O)C(NC(=O)OC(C=2C=CC=CC2N(=O)=O)C)C(=O)O
O=C(O)C1=C(O)C=C2C(=O)C=3C(O)=C(C(C(=O)C)=C(O)C3C(=O)C2=C1C(=O)O)CC
O=C(O)C=1C(Cl)=CC=C(C1C(=O)O)C(F)(F)F
O=C(O)C1=C(C(=O)O)C(=CC=C1C)C
O=C(O)C1=C(C(=O)O)C(=CC=C1N(=O)=O)N(=O)=O
O=C(O)C1=CC=CC(=C1C(=O)O)S(=O)(=O)N
O=C(O)C1=CC(N=NC=2C=C(C(=O)O)C(C(=O)O)=C(C2)C(=O)O)=CC(C(=O)O)=C1C(=O)O
O=C(O)C1=CC=CC(C=2C=CC=CC2C)=C1C(=O)O
O=C(O)C=1C(Br)=C(C(=O)O)C(C(=O)O)=C(Br)C1C(=O)O
O=C(O)C=1C(Br)=C(Br)C(Br)=C(Br)C1C(=O)O
O=C(O)C=1C(Br)=CC=C(Br)C1C(=O)O
O=C(O)C=1C=C(C(=O)O)C(C(=O)O)=C(C1C(=O)O)C(F)(F)F
O=C(O)C=1C=CC=C(NO)C1C(=O)O
O=C(O)C1=CC(Cl)=C(C(=O)O)C(C(=O)O)=C1Cl
O=C(O)C1=CC=C(C(=O)O)C(C(=O)O)=C1C(=O)O
O=C(O)C1=CC=C(C=C1)C=2C=CC=C(C(=O)O)C2C(=O)O
O=C(O)C1=C

In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\ver_4\out\csvfile\descriptors\gauss_train.csv")
df = df.sort_values("r2", ascending=False)
df.to_csv(r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\ver_4\out\csvfile\descriptors\1st_20260221.csv", index=False)

In [2]:
import pandas as pd

df = pd.read_csv(r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\ver_4\out\csvfile\predict\1st_20260204.csv")
for i in range(len(df)):
    print(df.loc[i, "smiles"])

O=C(O)C=1C=CC=CC1C(=O)O
O=C(O)C1=CC=CC(=C1C(=O)O)N(=O)=O
O=C(O)C1=CC=C(O)C=C1C(=O)O
O=C(O)C1=CC=C(C=C1C(=O)O)N(=O)=O
O=C(O)C1=CC=C(C(=O)O)C(=C1)C(=O)O
O=C(O)C=1C=C(C(=O)O)C(=CC1C(=O)O)C(=O)O
O=C(O)C1=CC=C(OC)C=C1C(=O)O
O=C(O)C1=CC=CC(C(=O)O)=C1C(=O)O
O=C(O)C1=CC=C(Br)C=C1C(=O)O
O=C(O)C=1C=CC=C(F)C1C(=O)O
O=C(O)C1=CC(Cl)=C(C(=O)O)C(C(=O)O)=C1Cl
O=C(O)C=1C(F)=CC=C(F)C1C(=O)O
O=C(O)C1=CC=C(F)C=C1C(=O)O
O=C(O)C=1C=CC=C(N)C1C(=O)O
O=C(O)C1=CC=2C=CC=CC2C=C1C(=O)O
O=C(O)C1=CC=C(C=C1C(=O)O)C=2C=CC(C(=O)O)=C(C2)C(=O)O
O=C(O)C=1C=CC=C(Br)C1C(=O)O
O=C(O)C=1C(O)=CC=C(O)C1C(=O)O
O=C(O)C1=CC=C(Cl)C=C1C(=O)O
O=C(O)C1=CC=C(C=C1C(=O)O)C
O=C(O)C=1C=CC=C(OC)C1C(=O)O
O=C(O)C1=CC=2N=CNC2C=C1C(=O)O
O=C(O)C=1C(Cl)=C(Cl)C(Cl)=C(Cl)C1C(=O)O
O=C(O)C1=CC(N)=CC(C(=O)O)=C1C(=O)O
O=C(O)C1=CC=C(C=C1C(=O)O)C(F)(F)F
O=C(O)C1=CC=CC(=C1C(=O)O)C
O=C(O)C1=CC=C(C=C1C(=O)O)C(N)C(=O)O
O=C(O)C=1C=C(C(=O)O)C(C(=O)O)=C(C(=O)O)C1C(=O)O
O=C(O)C=1C=CC=C(O)C1C(=O)O
O=C(O)C=1C=CC=C(Cl)C1C(=O)O
O=C(O)C=1C=C(C=C(C(=O)O)C1C(=O)O)N(=O)=

In [22]:
import pandas as pd

df1 = pd.read_csv(r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\ver_4\out\csvfile\known\2nd_20260205.csv")
# 24行目を選択
df1 = df1.iloc[24:25]
df2 = pd.read_csv(r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\ver_4\out\csvfile\unknown\unknown_data.csv")
df = pd.concat([df2, df1], ignore_index=True)
df.drop(columns=["yield"], inplace=True)
df.to_csv(r"C:\Users\kkyom\OneDrive\デスクトップ\pka_activity\ver_4\out\csvfile\unknown\unknown_data.csv", index=False)

In [1]:
θ = 0.5
print(θ)

0.5


In [4]:
base = ["C", "C", "C", "C"]
cn = "C#N"
no2 = "N(=O)=O"
base[0] = base[0] + "(" + cn + ")"
base[1] = base[1] + "(" + no2 + ")"
substruct = "OC(=O)C1=" + base[0] + base[1] + "=" + base[2] + base[3] + "=C1C(=O)O"
print(substruct)

OC(=O)C1=C(C#N)C(N(=O)=O)=CC=C1C(=O)O


In [25]:
functional_groups = {
    2: ("C#N", 0),
    "no2": ("N(=O)=O", 0),
    "f": ("F", 0),
    "cl": ("Cl", 0),
}

for i in functional_groups.keys():
    print(i)

2
no2
f
cl


In [22]:
functional_groups = {
    "cn": ("C#N", 0),
    "no2": ("N(=O)=O", 0),
    "f": ("F", 0),
    "cl": ("Cl", 0),
    "br": ("Br", 0),
    "h": ("", 0),
    "c6h5": ("C2=CC=CC=C2", 0),
    "ch3": ("C", 1),
    "oh": ("O", 1),
    "cho": ("C(=O)", 1),
    "cooh": ("C(=O)O", 1),
}

base = {
    "3": ["C", 1],
    "4": ["C", 1],
    "5": ["C", 1],
    "6": ["C", 1],
}

for c3 in functional_groups.items():
    base["3"][0] = "C"
    if c3[0] == "h":
        pass
    else:
        base["3"][0] += "(" + c3[1][0] + ")"
        base["3"][1] = c3[1][1]
    for c4 in functional_groups.items():
        base["4"][0] = "C"
        if c4[0] == "h":
            pass
        else:
            base["4"][0] += "(" + c4[1][0] + ")"
            base["4"][1] = c4[1][1]
        for c5 in functional_groups.items():
            base["5"][0] = "C"
            if c5[0] == "h":
                pass
            else:
                base["5"][0] += "(" + c5[1][0] + ")"
                base["5"][1] = c5[1][1]
            for c6 in functional_groups.items():
                base["6"][0] = "C"
                if c6[0] == "h":
                    pass
                else:
                    base["6"][0] += "(" + c6[1][0] + ")"
                    base["6"][1] = c6[1][1]
                print("OC(=O)C1=" + base["3"][0] + base["4"][0] + "=" + base["5"][0] + base["6"][0] + "=C1C(=O)O")

OC(=O)C1=C(C#N)C(C#N)=C(C#N)C(C#N)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(C#N)C(N(=O)=O)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(C#N)C(F)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(C#N)C(Cl)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(C#N)C(Br)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(C#N)C=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(C#N)C(C2=CC=CC=C2)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(C#N)C(C)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(C#N)C(O)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(C#N)C(C(=O))=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(C#N)C(C(=O)O)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(N(=O)=O)C(C#N)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(N(=O)=O)C(N(=O)=O)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(N(=O)=O)C(F)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(N(=O)=O)C(Cl)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(N(=O)=O)C(Br)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(N(=O)=O)C=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(N(=O)=O)C(C2=CC=CC=C2)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(N(=O)=O)C(C)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(N(=O)=O)C(O)=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(N(=O)=O)C(C(=O))=C1C(=O)O
OC(=O)C1=C(C#N)C(C#N)=C(N(=O)=O

In [ ]:
import datetime

now = datetime.date
print(now)


TypeError: descriptor 'strftime' for 'datetime.date' objects doesn't apply to a 'str' object

In [4]:
print("a")
display("a")
print(type(12))
display(type(12))

a


'a'

<class 'int'>


int

In [1]:
import datetime as dt
current = dt.date(2026, 3, 16)
print(current)

2026-03-16


In [2]:
current.test = "test"

AttributeError: 'datetime.date' object has no attribute 'test'

In [10]:
from rdkit import Chem

mol1 = Chem.MolFromSmiles("OC(=O)C1=CC=CC=C1C(=O)O")
mol2 = Chem.MolFromSmiles("COC(=O)C1=CC=CC=C1C(=O)O")
phthalic_pattern = Chem.MolFromSmarts("c1ccc(C(=O)[OH])c(C(=O)[OH])c1")

matches_list1 = mol1.GetSubstructMatches(phthalic_pattern)
matches_list2 = mol2.GetSubstructMatches(phthalic_pattern)
print(matches_list1)
print(matches_list2)
if matches_list1:
    print("mol1 has phthalic acid structure")
if matches_list2:
    print("mol2 has phthalic acid structure")

((5, 6, 7, 8, 9, 10, 11, 3, 1, 2, 0, 4),)
()
mol1 has phthalic acid structure


In [3]:
test = {
    1:1,
    2:2,
    3:3,
    4:4,
    5:5,
}
print(test)

{1: 1, 2: 2, 3: 3, 4: 4, 5: 5}


In [5]:
import itertools
a = list("abcd")
for i in itertools.combinations(a, 2):
    print(i)

('a', 'b')
('a', 'c')
('a', 'd')
('b', 'c')
('b', 'd')
('c', 'd')


In [37]:
from datetime import datetime, date

t = date.today().strftime("%Y%m%d")
print(t)

20260322


In [38]:
a = ("a", "b", "c")
print("|".join(a))

a|b|c


In [2]:
a = {1,3,4}
print(type(a))

<class 'set'>


In [7]:
dict1 = {"a": 1, "b": 2, "c": 3}
dict2 = {"a": 2, "b": 3, "c": 4}

result = []
result.append(dict1)
result.append(dict2)
print(result)

[{'a': 1, 'b': 2, 'c': 3}, {'a': 2, 'b': 3, 'c': 4}]


In [8]:
import pandas as pd
df = pd.DataFrame(result)
print(df)

   a  b  c
0  1  2  3
1  2  3  4


In [9]:
dict1 = {"d": 1, "e": 2, "f": 3}
dict2 = {"d": 2, "e": 3, "f": 4}

result = []
result.append(dict1)
result.append(dict2)

In [10]:
df[list(result[0].keys())] = pd.DataFrame(result, index=df.index)
print(df)

   a  b  c  d  e  f
0  1  2  3  1  2  3
1  2  3  4  2  3  4


In [12]:
result[0].keys()

dict_keys(['d', 'e', 'f'])

In [4]:
from pathlib import Path

base = Path("C:\Users\kyoma\Desktop\laboratory\ml\predict_activity\ver_4")
print(base)

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (640137429.py, line 3)

In [1]:
import datetime
print(type(datetime.date.today().strftime("%Y%m%d")))

<class 'str'>
